In [ ]:
#!pip install langchain langchain_openai langgraph -q

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxx'

In [ ]:
model = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
class SentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')


In [ ]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [ ]:
prompt = 'What is the sentiment of the following review - The software too good'
structured_model.invoke(prompt).sentiment

In [ ]:
# 1. UX issue, calm, low urgency
prompt1 = """
Analyze the following user review and extract the issue type, tone, and urgency:

Review: "The interface is a bit cluttered. It would be nice if the menus were more organized."
"""

# 2. Performance issue, frustrated, medium urgency
prompt2 = """
Analyze the following user review and extract the issue type, tone, and urgency:

Review: "The app runs really slowly on my phone. It’s annoying because I use it daily."
"""

# 3. Bug, angry, high urgency
prompt3 = """
Analyze the following user review and extract the issue type, tone, and urgency:

Review: "Every time I try to log in, the app crashes. This is unacceptable — fix it now!"
"""

# 4. Support-related, disappointed, medium urgency
prompt4 = """
Analyze the following user review and extract the issue type, tone, and urgency:

Review: "I contacted support a week ago and still haven’t received a reply. Really disappointed with the lack of response."
"""

# 5. Other, calm, low urgency
prompt5 = """
Analyze the following user review and extract the issue type, tone, and urgency:

Review: "I wish the app had dark mode. Not a big deal, but it would be a nice addition."
"""
result = structured_model2.invoke(prompt2)
print('issue_type: ', result.issue_type)
print('tone : ', result.tone)
print('urgency : ',result.urgency)

In [ ]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}


In [ ]:
def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'


In [ ]:
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""

    response = model.invoke(prompt).content

    return {'response': response}




In [ ]:
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

In [ ]:
def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
outcome = workflow.invoke(intial_state)

In [ ]:
outcome

In [ ]:
print(outcome['response'])